# COD Data Mining: Identifying Interesting Crystal Structures
### Teaching Tool for Crystallography

This notebook demonstrates how to query the **Crystallography Open Database (COD)** to identify structures that may deserve closer human inspection. 

**Pedagogical Goal:** To understand that automated filters find *discussion candidates*, not definitive answers. All scientific judgment happens *after* reading the paper and checking the data.

---

## Step 1: Setup and Helper Functions

We use a helper function to fetch data from COD. 
*Note: We handle cases where no results are found to avoid errors, and we rename technical column names for clarity.*

In [1]:
import pandas as pd

def run_cod_query(query_string):
    """
    Runs a search on COD and returns a pandas DataFrame.
    Handles empty results and renames columns for readability.
    """
    base_url = "https://www.crystallography.net/cod/result?"
    url = base_url + query_string
    
    print(f"Querying COD with parameters: {query_string}\n")
    
    try:
        # COD CSVs start with comment lines (#)
        df = pd.read_csv(url, comment='#')
    except pd.errors.EmptyDataError:
        # If no results are found, COD returns only comments, which pandas sees as empty.
        print("No results found for this query.\n")
        return pd.DataFrame(columns=['cod_id', 'formula', 'cell_volume', 'z', 'zprime', 'spacegroup'])
    
    # Rename columns to be more descriptive
    rename_map = {
        'file': 'cod_id',
        'vol': 'cell_volume',
        'sg': 'spacegroup',
        'Z': 'z',
        'Zprime': 'zprime'
    }
    df = df.rename(columns=rename_map)
    
    return df

## Task 1: Structures with Very Large Unit Cells

**Rationale:** Very large unit-cell volumes for chemically simple systems may indicate missed symmetry, poor modeling, or unmodelled solvent.

In [2]:
# Search for structures with volume >= 5000 Å^3 and at most 3 different elements
query_large_cells = (
    "format=csv&"       # Return CSV
    "vmin=5000&"        # Minimum volume
    "vmax=1000000&"     # Maximum volume (must be large to avoid [5000, 0] range)
    "strictmin=1&"      # At least 1 element
    "strictmax=3"       # No more than 3 different elements
)

df_large_cells = run_cod_query(query_large_cells)

if not df_large_cells.empty:
    # Show the 10 largest cells found
    display(df_large_cells.sort_values("cell_volume", ascending=False)[[
        "cod_id", "formula", "cell_volume", "z", "spacegroup"
    ]].head(10))

Querying COD with parameters: format=csv&vmin=5000&vmax=1000000&strictmin=1&strictmax=3



,cod_id,formula,cell_volume,z,spacegroup
401,1550918,- O4 S20 Sn10 -,398948.0,136.0,F d -3 m
837,2104375,- Al12827.56 Cu1244.05 Ta9063 -,365373.0,1.0,F -4 3 m
419,1552091,- O5280 Si2640 -,167011.0,1.0,I m -3 m
2428,7125310,- O805 Sr48 U96 -,154750.0,8.0,F m -3 c
978,2311717,- Cu3.311 Si -,123871.0,2352.0,P -3
853,2105953,- Al67.4 Ni14.7 Rh17.8 -,94072.5,1.0,P 1
836,2104374,- Al3339.8 Cu231.95 Ta2336 -,93428.0,1.0,F -4 3 m
420,1552092,- O2880 Si1440 -,91550.9,1.0,I m -3 m
600,1570609,- Na156 O2880 Si1440 -,81008.0,1.0,I -4 3 m
1015,4003639,- C360 O91 Zn28 -,76287.0,1.0,P -4 3 m


## Task 2: High Z' (Z-prime) Organic Structures

**Rationale:** High Z' structures have many independent molecules in the asymmetric unit. While not inherently wrong, they are statistically over-represented among problematic refinements.

In [3]:
# Search for organic structures (containing Carbon) with Z' >= 3
query_high_zprime = (
    "format=csv&"
    "minZprime=3&"      # Z' >= 3
    "el1=C"             # Must contain Carbon
)

df_high_zprime = run_cod_query(query_high_zprime)

if not df_high_zprime.empty:
    display(df_high_zprime[[
        "cod_id", "formula", "zprime", "cell_volume", "spacegroup"
    ]].head(10))

Querying COD with parameters: format=csv&minZprime=3&el1=C



,cod_id,formula,zprime,cell_volume,spacegroup
0,1100404,- C26 H27 Li O2 -,4.0,2192.74,P 1
1,1100526,- C10 H10 As2 Ti -,4.0,1981.56,P -1
2,1100571,- C22 H39 O6.5 -,4.0,4469.25,P -1
3,1100895,- C24 H47 Cl N5 Nb O Si2 -,4.0,12960.80,P 1 21/n 1
4,1100907,- C17 H15 N O4 -,3.0,4357.90,P 21 21 21
5,1101055,- C20 H14 O2 -,4.0,8729.70,P 63
6,1101134,- C15 H31.75 Cl0.88 K0.88 N3 O8.38 Pd3 S3 -,4.0,2853.70,P 1
7,1101141,- C5 H8 Cu N3 O5.5 -,4.0,4084.40,P 21 21 21
8,1501605,- C10 H18 N10 O -,3.0,2047.50,P -1
9,1501784,- C14 H20 Cl2 N2 Ti -,3.0,4801.26,P 1 21/c 1


## Task 3: Structures in the P-1 Space Group

**Rationale:** P-1 is often correct, but it is also a "space group of last resort." These structures deserve careful symmetry inspection.

In [4]:
# Search for organic structures in space group P-1 (Space Group Number 2)
query_p1 = (
    "format=csv&"
    "space_group_number=2&" # Robust way to search for P-1
    "el1=C"
)

df_p1 = run_cod_query(query_p1)

if not df_p1.empty:
    display(df_p1[[
        "cod_id", "formula", "cell_volume", "z", "zprime"
    ]].head(10))

Querying COD with parameters: format=csv&space_group_number=2&el1=C



/var/folders/h3/4snxgkvd30938m00kt306pnm0000gn/T/ipykernel_32831/3650491260.py:15: DtypeWarning: Columns (23,24,32,38,44,45,46,52,67) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(url, comment='#')


,cod_id,formula,cell_volume,z,zprime
0,1000004,- C29 H30 Cu I P2 -,1364.900,2.0,1.0
1,1000506,- C12 H48 Al7 F29 N8 O2 -,877.962,1.0,0.5
2,1000507,- C12 H47 Al7 F30 N8 O -,915.903,1.0,0.5
3,1000510,- C23 H34 O4 Si -,1172.520,2.0,1.0
4,1004000,- C6 H30 N6 S8 W6 -,619.100,1.0,0.5
5,1004002,- C114 H96 P6 S8 W6 -,5111.000,2.0,1.0
6,1004005,- C54 H78 N6 S8 W6 -,1663.410,1.0,0.5
7,1004028,- C10 H30 Co4 Ga6 N4 O41 P10 -,1321.200,1.0,0.5
8,1004032,- C12 H43 F6 Ga5 N4 O22 P4 -,838.900,1.0,0.5
9,1007224,- C2 H20 Cu Li2 N6 O22 P6 -,585.000,1.0,0.5


## Step 2: Classroom Discussion

1.  **Select a Row:** Pick an interesting COD ID from the results above.
2.  **Download & Check:** Download the CIF from [crystallography.net](https://www.crystallography.net) using the ID.
3.  **Validate:** Run [CheckCIF](http://checkcif.iucr.org) and read the original paper.
4.  **Decide:** Is the structure trustworthy? Does the data support the published interpretation?

---